In [ ]:
# check the machine

!nproc
!free -g | head -2
!pip list 2>/dev/null | grep -i -E "^(torch|transformers|safetensors|opencv)"
!ls /kaggle/input/datasets/saisdsaiiitbh/

In [2]:
!df -h /kaggle/working /kaggle/temp /tmp 2>/dev/null
!free -g | head -2

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G   76K   20G   1% /kaggle/working
overlay         8.0T  7.0T  1.1T  88% /
               total        used        free      shared  buff/cache   available
Mem:              31           1          25           0           4          30


In [4]:
!cp /kaggle/input/datasets/saisdsaiiitbh/dff-code/*.py /kaggle/working/
%cd /kaggle/working
!python test_dff_adapter.py --real

/kaggle/working
PASS  test_adapters_wrap_q_v_dense_in_every_block
PASS  test_each_input_chunk_used_exactly_once
PASS  test_gradient_flow_and_unselected_logits
PASS  test_linear_probe_variant
PASS  test_lora_baseline_variant
PASS  test_matches_naive_equations
PASS  test_only_adapters_and_heads_train
      Small  adapter params =    665,064
      Base   adapter params =  1,328,616
      Large  adapter params =  3,541,968
PASS  test_parameter_budget_matches_paper
config.json: 100%|█████████████████████████████| 547/547 [00:00<00:00, 2.07MB/s]

model.safetensors: downloading bytes: █████████████████▊    | 71.4MB, 5.56MB/s  
model.safetensors: downloading bytes: ██████████████████████| 83.7MB, 7.99MB/s  
model.safetensors: reconstructing file: 100%|██████| 88.2MB / 88.2MB, 8.56MB/s  
Loading weights: 100%|██████████████████████| 223/223 [00:00<00:00, 2732.19it/s]
      trainable=667,374 of total=22,723,950 (2.94%)
PASS  test_real_dinov2_small
      never-selected (adapter,head,expert) slots

In [6]:
import os, shutil, time
from concurrent.futures import ThreadPoolExecutor
BASE = "/kaggle/input/datasets/saisdsaiiitbh"
TMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
print("local scratch dir:", TMP)
def copy_tree(src, dst, threads=32):
    files = [os.path.join(r, n) for r, _, ns in os.walk(src) for n in ns]
    print(f"{src}: {len(files)} files", flush=True)
    def cp(p):
        q = os.path.join(dst, os.path.relpath(p, src))
        os.makedirs(os.path.dirname(q), exist_ok=True)
        shutil.copyfile(p, q)
    t = time.time()
    with ThreadPoolExecutor(threads) as ex:
        for i, _ in enumerate(ex.map(cp, files)):
            if i % 20000 == 0: print(f"  {i} files, {time.time()-t:.0f}s", flush=True)
    print(f"  done in {time.time()-t:.0f}s", flush=True)
copy_tree(f"{BASE}/eval-corps", f"{TMP}/ev")
copy_tree(f"{BASE}/ff-crops",   f"{TMP}/ff")

local scratch dir: /tmp
/kaggle/input/datasets/saisdsaiiitbh/eval-corps: 0 files
  done in 0s
/kaggle/input/datasets/saisdsaiiitbh/ff-crops: 0 files
  done in 0s


In [7]:
!find /kaggle/temp/ff /tmp/ff -name "*.jpg" 2>/dev/null | wc -l
!find /kaggle/temp/ff /tmp/ff -name "_status.json" 2>/dev/null | wc -l

0
0


In [8]:
!python evaluate_dff.py --ckpt runs/smoke_dff_full/best.pt --backbone small \
  --sets /kaggle/temp/ff/ffpp_manifest_split.csv:val /kaggle/temp/ev/cdf1_manifest.csv:all \
  --frames 4 --num_workers 4 --out /kaggle/working/smoke_eval.json

Loading weights: 100%|██████████████████████| 223/223 [00:00<00:00, 2672.85it/s]
Traceback (most recent call last):
  File "/kaggle/working/evaluate_dff.py", line 146, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/evaluate_dff.py", line 128, in main
    load_checkpoint(model, args.ckpt)
    ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/train.py", line 71, in load_checkpoint
    ck = torch.load(path, map_location="cpu", weights_only=False)
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 1530, in load
    with _open_file_like(f, "rb") as opened_file:
         ~~~~~~~~~~~~~~~^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 795, in _open_file_like
    return _open_file(name_or_buffer, mode)
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 776, in __init__
    super().__init__(open(name, mode))  # noqa: SIM115
                     ~~~~^^^^^^^^^^^^
FileNotFoundError: [Err

In [9]:
%env FF_ROOT=/kaggle/temp/ff
%env EV_ROOT=/kaggle/temp/ev
!python -u train.py --backbone small --crops_root /kaggle/temp/ff --out_dir runs/diag \
  --epochs 1 --frames_per_video 2 --batch_size 24 --num_workers 4 --log_every 20 --val_frames 2

env: FF_ROOT=/kaggle/temp/ff
env: EV_ROOT=/kaggle/temp/ev
Traceback (most recent call last):
  File "/kaggle/working/train.py", line 235, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/train.py", line 138, in main
    train_ds = FrameDataset(args.crops_root, manifest, "train", "train", args.frames_per_video)
  File "/kaggle/working/dataset.py", line 77, in __init__
    rows = read_manifest(manifest, split)
  File "/kaggle/working/dataset.py", line 31, in read_manifest
    rows = list(csv.DictReader(open(path)))
                               ~~~~^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/temp/ff/ffpp_manifest_split.csv'
